# Agentic Harness Config Generator

Takes API providers and their API endpoints and generates a Harness config file for the Agentic Harnes

Currently supports the following formats:

- Hermes Agent config\.yml

- Pi Coding Agent models\.json

- LiteLLM config\.yaml

## Fetch models

Obtain list of the models by providing an OpenAI compatible endpoint URL and your API key\. Alternatively, you can upload a JSON with the models directly

In [ ]:
api_endpoint = ''

In [ ]:
api_key = ''

In [ ]:
api_key_env = ''

In [ ]:
models_upload = None

In [ ]:
import json
import os
import requests

endpoint = api_endpoint.strip() if 'api_endpoint' in globals() and api_endpoint else ""
key = api_key.strip() if 'api_key' in globals() and api_key else ""
uploaded_file = models_upload if 'models_upload' in globals() and models_upload else None

# Rule 1: Endpoint AND Key present -> Prioritize API fetch regardless of file upload
if endpoint and key:
    print(f"[API Fetch] Prioritizing API endpoint fetch from: {endpoint}")
    url = f"{endpoint.rstrip('/')}/models"
    headers = {"Authorization": f"Bearer {key}"}
    
    res = requests.get(url, headers=headers, timeout=15)
    res.raise_for_status()  # Fail fast on HTTP errors
    data = res.json()       # Fail fast on non-JSON response
    
    active_models_file = "models.json"
    with open(active_models_file, "w") as f:
        json.dump(data, f)
    print(f"[API Fetch] Successfully fetched models and saved to {active_models_file}")

# Rule 2: Endpoint present, NO key, BUT file upload present -> Fallback to upload
elif endpoint and not key and uploaded_file:
    if not os.path.exists(uploaded_file):
        raise FileNotFoundError(f"Uploaded file does not exist: {uploaded_file}")
    if not uploaded_file.lower().endswith(".json"):
        raise ValueError(f"Uploaded file must be a .json file, received: {uploaded_file}")
    
    with open(uploaded_file, "r") as f:
        data = json.load(f)  # Fail fast on corrupted JSON
    
    active_models_file = uploaded_file
    print(f"[Fallback] EXPLICIT NOTE: No API key provided for {endpoint}. Falling back to uploaded file: {active_models_file}")

# Standalone File Upload (No endpoint set)
elif not endpoint and uploaded_file:
    if not os.path.exists(uploaded_file):
        raise FileNotFoundError(f"Uploaded file does not exist: {uploaded_file}")
    if not uploaded_file.lower().endswith(".json"):
        raise ValueError(f"Uploaded file must be a .json file, received: {uploaded_file}")
    
    with open(uploaded_file, "r") as f:
        data = json.load(f)  # Fail fast on corrupted JSON
    
    active_models_file = uploaded_file
    print(f"[File Upload] Using uploaded JSON file: {active_models_file}")

# Fail Fast on all unhandled / invalid input combinations
else:
    if endpoint and not key and not uploaded_file:
        raise ValueError("API endpoint provided without an API key, and no fallback JSON file was uploaded.")
    raise ValueError("Invalid configuration: Must provide both API Endpoint + API Key OR upload a valid JSON file.")


## Load and initialize state

In [ ]:
# Initialize

### Analyze model schema

This will show how the provider's schema for the model so that it is clear\. For use with table configuration and overrides

In [ ]:
# analyze model schema

### Configure table

This section allows for better configuration when displaying models\. While OpenAI compatible format standardizes id, object, created, and owned\_by\. But there are provider discrepancies that must be addressed to properly map them to the actual agentic config

## Search Parameters and overrides

In [ ]:
regex_pattern = '*'

In [ ]:
page_size = '10'

In [ ]:
page_number = 1

## Find models

In [ ]:
# Filter by Regex
if regex_pattern:
    try:
        filtered_df = df[df['id'].str.contains(regex_pattern, case=False, na=False, regex=True)]
    except Exception:
        filtered_df = df
else:
    filtered_df = df

# Slice page
start_idx = (page_num - 1) * page_size
end_idx = start_idx + page_size
page_df = filtered_df.iloc[start_idx:end_idx]

print(f"Showing {len(page_df)} of {len(filtered_df)} matching models (Total: {len(df)})")
display(page_df[['id', 'context_length', 'pricing']])


## Pin Models

In [ ]:
pin_model_id = ''

In [ ]:
button_1 = False

In [ ]:
# NOTE: variable names currently don't match

# Append model ID if typed and not already pinned
if pin_id and pin_id in df['id'].values and pin_id not in pinned_models:
    pinned_models.append(pin_id)

print(f"Currently Pinned Models ({len(pinned_models)}):")
if pinned_models:
    display(df[df['id'].isin(pinned_models)][['id', 'context_length', 'pricing']])


## Generate Configs

### Hermes Agent

Generates configs for Hermes Agent. Requires 2 files to be updated: `config.yml` and `.env`

### Pi Coding Agent

### LiteLLM

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=cf941940-56b9-494d-b9e4-8b53a91f8a1e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>